# MJX 06 — Reinforcement Learning Refresher

### Lab Description

Before training policies on the GPU, a quick refresher on the vocabulary, grounded in a real MJX task (`inverted_pendulum` from Brax — keep the cart's pole upright).

- **Environment / MDP**: state `s`, action `a`, transition `s' = f(s, a)`, reward `r(s, a)`.
- **Observation** `obs`: what the agent sees each step (here: cart/pole positions & velocities).
- **Policy** `π(a | obs)`: maps observations to actions.
- **Return**: the (discounted) sum of future rewards; the **value** estimates the expected return.
- **Goal of RL**: find a policy that maximizes expected return.

Here we just *run* hand-written policies to see how the pieces fit; the next notebooks *learn* a good policy automatically.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Load a Brax/MJX environment and inspect its observation/action sizes
- Roll out a policy and accumulate reward (return)
- Compare a random policy against a hand-tuned controller
- Render both policies side by side to see success vs. failure

### Load the environment

`brax.envs.get_environment("inverted_pendulum", backend="mjx")` gives a GPU (MJX) environment. We jit its `reset` and `step`. The observation is `[cart position, pole angle, cart velocity, pole angular velocity]` (size 4); the action is a single cart force.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # hide Brax's "not maintained" notice

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jp
import brax.envs as be

env = be.get_environment("inverted_pendulum", backend="mjx")
print("observation size:", env.observation_size, "| action size:", env.action_size)
reset = jax.jit(env.reset)
step = jax.jit(env.step)

### Run two policies

`run_episode` steps a policy until the pole falls (`state.done`) or a step cap. We compare:
- **random** — uniform random forces (a baseline that learns nothing).
- **heuristic** — a hand-tuned PD controller: push the cart toward the side the pole leans (`obs[1]`) and damp its angular velocity (`obs[3]`). The *sign* matters — pushing the wrong way makes it fall faster.

Reward is `+1` per surviving step, so the return equals how long the pole stayed up.

In [ ]:
# Roll out a RANDOM policy for one episode and watch the reward.
def run_episode(policy, n=300, seed=0):
    rng = jax.random.PRNGKey(seed)
    state = reset(rng)
    rewards = []
    for _ in range(n):
        rng, k = jax.random.split(rng)
        action = policy(state.obs, k)
        state = step(state, action)
        rewards.append(float(state.reward))
        if state.done:
            break
    return np.array(rewards)

def random_policy(obs, key):
    return jax.random.uniform(key, (env.action_size,), minval=-1.0, maxval=1.0)

# A hand-tuned PD controller: push the cart toward the side the pole is
# leaning (obs[1] = pole angle) and damp the pole's angular velocity
# (obs[3]). The sign matters — pushing the wrong way makes it fall faster.
def heuristic_policy(obs, key):
    return jp.array([50.0 * obs[1] + 2.0 * obs[3]]).clip(-1.0, 1.0)

r_rand = run_episode(random_policy)
r_heur = run_episode(heuristic_policy)
print(f"random policy:    survived {len(r_rand):3d} steps, return = {r_rand.sum():.1f}")
print(f"heuristic policy: survived {len(r_heur):3d} steps, return = {r_heur.sum():.1f}")

### Plot the returns

Cumulative reward over time *is* the survival curve. The random policy's short stub is drawn on top with markers so it stays visible next to the much longer heuristic curve.

In [ ]:
# Reward is +1 per surviving step, so the return curve == how long the pole
# stays up. Draw random on top with markers so its short stub is visible
# where it would otherwise hide under the heuristic line.
plt.figure(figsize=(7, 3))
plt.plot(np.cumsum(r_heur), lw=2, color="C1", label=f"heuristic (return {r_heur.sum():.0f})")
plt.plot(np.cumsum(r_rand), "o-", lw=2, ms=6, color="C0", zorder=3, label=f"random (return {r_rand.sum():.0f})")
plt.xlabel("step"); plt.ylabel("cumulative reward (return)"); plt.title("Inverted pendulum: random vs heuristic policy")
plt.legend(); plt.grid(True); plt.show()

### Watch them — random falls, heuristic balances

Numbers are clearer as motion. We roll out both policies for the same 150 steps (this time *without* stopping at episode end, so the failure is visible) and render them side by side. **Left: random** (the pole topples and lies flat). **Right: heuristic** (the controller keeps it upright).

In [ ]:
import os, mujoco, imageio
os.makedirs("output/videos", exist_ok=True)

# Brax's default renderer draws on a black background with a brown pole. We
# render with mujoco.Renderer instead and (a) recolour the geoms and (b)
# composite a soft gradient background using a segmentation mask, for a much
# nicer-looking video.
m = env.sys.mj_model
m.geom_rgba[0] = [0.82, 0.82, 0.86, 1]   # rail  -> light grey
m.geom_rgba[1] = [0.30, 0.34, 0.42, 1]   # cart  -> slate
m.geom_rgba[2] = [0.10, 0.55, 0.85, 1]   # pole  -> blue
d = mujoco.MjData(m)

H, W = 240, 320
_top, _bot = np.array([60, 78, 110]), np.array([200, 212, 228])
_grad = (_top * (1 - np.linspace(0, 1, H))[:, None] + _bot * np.linspace(0, 1, H)[:, None]).astype(np.uint8)
BG = np.repeat(_grad[:, None, :], W, axis=1)  # soft blue-grey vertical gradient

def render_states(states):
    R = mujoco.Renderer(m, height=H, width=W)
    out = []
    for ps in states:
        d.qpos[:] = np.asarray(ps.q); mujoco.mj_forward(m, d)
        R.update_scene(d, camera=-1); rgb = R.render()
        # segmentation: pixels touching no geom (id < 0) are background
        R.enable_segmentation_rendering(); R.update_scene(d, camera=-1)
        seg = R.render()[:, :, 0]; R.disable_segmentation_rendering()
        out.append(np.where((seg < 0)[:, :, None], BG, rgb).astype(np.uint8))
    R.close()
    return np.asarray(out)

def render_rollout(policy, n=150, seed=0):
    rng = jax.random.PRNGKey(seed)
    state = reset(rng)
    states = [state.pipeline_state]
    for _ in range(n):  # keep stepping (ignore done) so the pole's fall is visible
        rng, k = jax.random.split(rng)
        state = step(state, policy(state.obs, k))
        states.append(state.pipeline_state)
    return states

N = 150
f_rand = render_states(render_rollout(random_policy, N))
f_heur = render_states(render_rollout(heuristic_policy, N))
sep = np.full((f_rand.shape[0], f_rand.shape[1], 4, 3), 255, np.uint8)  # white divider
combo = np.concatenate([f_rand, sep, f_heur], axis=2)  # left = random, right = heuristic
imageio.mimsave("output/videos/mjx06_random_vs_heuristic.mp4", list(combo), fps=30)
print("saved side-by-side video:", combo.shape[0], "frames")

### Watch the comparison

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx06_random_vs_heuristic.mp4")

## Conclusions

Even a crude hand-tuned policy keeps the pole up far longer than random actions — but tuning it by hand was fiddly and the sign was easy to get wrong. In MJX 07 we replace hand-tuning with **PPO**, which learns a much better policy automatically by training over hundreds of GPU-parallel environments.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT